# Molecular Graph Learning Curve

This notebook measures how predictive performance on the MoleculeNet HIV dataset changes as the number of streamed training graphs increases, using `NSPPK` features with `radius=1`, `distance=4`, `connector=1` and a linear `SGDClassifier`.

The first `TEST_PREFIX_SIZE` molecules are materialized, balanced, and used as a fixed test set. Training is then repeated on random Bernoulli samples from the remaining molecules with `LIMIT = 0.9`, so each repeat sees a different large training stream while the same held-out test set is scored at exponentially growing train sizes. Because the default HIV setup does not require learned attribute preprocessing, the benchmark can stream labeled batches directly without a separate warmup fit phase.


In [1]:
from pathlib import Path
import gc
import os
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import RDLogger
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

try:
    import psutil
except ImportError:
    psutil = None

REPO_CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
REPO_ROOT = next(candidate for candidate in REPO_CANDIDATES if (candidate / 'src').exists())
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from nsppk import NSPPK
from utils import plot_series_with_band_loess

DATASET_FILE = REPO_ROOT / 'data' / 'HIV.csv'
NBIT = 14
RADIUS = 1
DISTANCE = 4
CONNECTOR = 1
LIMIT = 0.99
TEST_PREFIX_SIZE = 2000
BALANCE_TEST_SET = True
TRAIN_SIZE_VALUES = None
N_REPEATS = 3
BATCH_SIZE = 256
PARALLEL = True
RANDOM_STATE = 42

RDLogger.DisableLog('rdApp.*')

def current_rss_mb():
    if psutil is not None:
        return psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)
    import resource
    rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    if sys.platform == 'darwin':
        return rss / (1024 ** 2)
    return rss / 1024

MEMORY_LOG = []
MEMORY_PEAK_MB = 0.0

def record_memory(stage, emit=True):
    global MEMORY_PEAK_MB
    rss_mb = float(current_rss_mb())
    MEMORY_PEAK_MB = max(MEMORY_PEAK_MB, rss_mb)
    event = {'stage': stage, 'rss_mb': rss_mb, 'peak_rss_mb': MEMORY_PEAK_MB}
    MEMORY_LOG.append(event)
    if emit:
        print(f"[memory] {stage:<28} rss={rss_mb:8.2f} MB  peak={MEMORY_PEAK_MB:8.2f} MB")
    return event

def count_dataset_rows(dataset_file):
    with Path(dataset_file).open() as stream:
        return max(sum(1 for _ in stream) - 1, 0)

def resolve_limit_count(limit, available_count):
    if limit is None:
        return int(available_count)
    if isinstance(limit, int):
        return min(int(limit), int(available_count))
    return min(int(np.floor(float(limit) * available_count)), int(available_count))

def make_halving_train_sizes(dataset_size, train_limit, min_size=BATCH_SIZE):
    if train_limit <= 0:
        return []
    if TRAIN_SIZE_VALUES is not None:
        return [n for n in TRAIN_SIZE_VALUES if n <= train_limit]
    train_sizes = []
    current = dataset_size // 2
    while current >= min_size:
        if current <= train_limit:
            train_sizes.append(current)
        current //= 2
    return sorted(set(train_sizes))


## Setup

Build the fixed balanced test set from the first `TEST_PREFIX_SIZE` molecules, instantiate the vectorizer directly, and convert the fractional `LIMIT` into an approximate sampled training-pool size. The checkpoint schedule is then derived by repeatedly halving the total dataset size down to a minimum of one `BATCH_SIZE`, keeping only values that fit within the sampled training cap.


In [2]:
loader = NSPPK(radius=RADIUS, distance=DISTANCE, connector=CONNECTOR, nbits=NBIT, parallel=PARALLEL)
record_memory('start')

test_graphs = loader.load_from(
    DATASET_FILE,
    'smiles',
    limit=TEST_PREFIX_SIZE,
    balance=BALANCE_TEST_SET,
    random_state=RANDOM_STATE,
    label_extractor=lambda graph: int(graph.graph['HIV_active']),
)
y_test = np.asarray([int(graph.graph['HIV_active']) for graph in test_graphs])
record_memory('test set loaded')

dataset_size = count_dataset_rows(DATASET_FILE)
available_train_pool = max(dataset_size - TEST_PREFIX_SIZE, 0)
train_limit = resolve_limit_count(LIMIT, available_train_pool)

vectorizer = NSPPK(
    radius=RADIUS,
    distance=DISTANCE,
    connector=CONNECTOR,
    nbits=NBIT,
    dense=False,
    parallel=PARALLEL,
)
record_memory('vectorizer ready')
X_test = vectorizer.transform(test_graphs)
record_memory('test transformed')

classes = np.array([0, 1])
class_weight = None

train_size_values = make_halving_train_sizes(dataset_size, train_limit)

print('test graphs:', len(test_graphs))
print(f'test positive rate: {y_test.mean():.3f}')
print('dataset size:', dataset_size)
print('available train pool:', available_train_pool)
print('train pool limit:', train_limit)
print('class weight for partial_fit:', class_weight)
print('stream batch size:', BATCH_SIZE)
print('train sizes:', train_size_values)
print(f'initial peak RSS: {MEMORY_PEAK_MB:.2f} MB')


[memory] start                        rss=  265.43 MB  peak=  265.43 MB
[memory] test set loaded              rss= 1169.79 MB  peak= 1169.79 MB
[memory] vectorizer ready             rss= 1169.79 MB  peak= 1169.79 MB
[memory] test transformed             rss= 1242.91 MB  peak= 1242.91 MB
test graphs: 2000
test positive rate: 0.500
dataset size: 82255
available train pool: 80255
train pool limit: 79452
class weight for partial_fit: None
stream batch size: 256
train sizes: [321, 642, 1285, 2570, 5140, 10281, 20563, 41127]
initial peak RSS: 1242.91 MB


## Incremental Learning Curve

Train `SGDClassifier` with `partial_fit(...)` on successive labeled batches drawn from a repeat-specific Bernoulli sample of the post-test dataset. The first streamed batch initializes `partial_fit(...)` with the full class list, and later batches continue incrementally. Metrics and memory are recorded whenever the cumulative trained count crosses one of the power-of-two checkpoints generated from the sampled training limit.


In [ ]:
results = []

print(f"{'train':>8} | {'repeat':>6} | {'roc_auc':>8} | {'avg_prec':>8} | {'seconds':>8} | {'rss_mb':>8} | {'peak_mb':>8}")
print('-' * 79)

repeat_histories = {
    train_size: {'roc_auc': [], 'avg_precision': [], 'runtime_sec': [], 'rss_mb': [], 'peak_rss_mb': []}
    for train_size in train_size_values
}

for repeat_idx in range(1, N_REPEATS + 1):
    classifier = SGDClassifier(
        loss='log_loss',
        alpha=1e-5,
        penalty='l2',
        random_state=RANDOM_STATE + repeat_idx,
        class_weight=class_weight,
    )

    t0 = time.perf_counter()
    trained_count = 0
    checkpoint_idx = 0
    is_first_batch = True

    def score_reached_checkpoints(trained_count, checkpoint_idx):
        while checkpoint_idx < len(train_size_values) and trained_count >= train_size_values[checkpoint_idx]:
            stop = train_size_values[checkpoint_idx]
            memory_event = record_memory(f'repeat {repeat_idx} train {stop}', emit=False)
            y_score = classifier.decision_function(X_test)
            runtime_sec = time.perf_counter() - t0
            roc_auc = roc_auc_score(y_test, y_score)
            avg_precision = average_precision_score(y_test, y_score)

            repeat_histories[stop]['roc_auc'].append(float(roc_auc))
            repeat_histories[stop]['avg_precision'].append(float(avg_precision))
            repeat_histories[stop]['runtime_sec'].append(float(runtime_sec))
            repeat_histories[stop]['rss_mb'].append(float(memory_event['rss_mb']))
            repeat_histories[stop]['peak_rss_mb'].append(float(memory_event['peak_rss_mb']))

            print(
                f"{stop:>8d} | {repeat_idx:>6d} | {roc_auc:>8.4f} | {avg_precision:>8.4f} | "
                f"{runtime_sec:>8.2f} | {memory_event['rss_mb']:>8.2f} | {memory_event['peak_rss_mb']:>8.2f}"
            )
            checkpoint_idx += 1

        return checkpoint_idx

    for X_batch, y_batch in vectorizer.stream_from(
        DATASET_FILE,
        'smiles',
        limit=LIMIT,
        random_state=RANDOM_STATE + repeat_idx,
        batch_size=BATCH_SIZE,
        label_extractor=lambda graph: int(graph.graph['HIV_active']),
        start_after_instance=TEST_PREFIX_SIZE,
    ):
        if is_first_batch:
            classifier.partial_fit(X_batch, y_batch, classes=classes)
            is_first_batch = False
        else:
            classifier.partial_fit(X_batch, y_batch)
        trained_count += len(y_batch)
        checkpoint_idx = score_reached_checkpoints(trained_count, checkpoint_idx)
        del X_batch, y_batch
        gc.collect()

    print(f'repeat {repeat_idx} sampled train graphs: {trained_count}')

reached_train_sizes = []
skipped_train_sizes = []
for train_size in train_size_values:
    history = repeat_histories[train_size]
    if not history['roc_auc']:
        skipped_train_sizes.append(train_size)
        continue
    reached_train_sizes.append(train_size)
    results.append(
        {
            'train_size': train_size,
            'n_repeats_reached': len(history['roc_auc']),
            'mean_roc_auc': float(np.mean(history['roc_auc'])),
            'std_roc_auc': float(np.std(history['roc_auc'], ddof=0)),
            'mean_avg_precision': float(np.mean(history['avg_precision'])),
            'std_avg_precision': float(np.std(history['avg_precision'], ddof=0)),
            'mean_runtime_sec': float(np.mean(history['runtime_sec'])),
            'std_runtime_sec': float(np.std(history['runtime_sec'], ddof=0)),
            'mean_rss_mb': float(np.mean(history['rss_mb'])),
            'mean_peak_rss_mb': float(np.mean(history['peak_rss_mb'])),
            'nbits': NBIT,
            'radius': RADIUS,
            'distance': DISTANCE,
            'connector': CONNECTOR,
        }
    )

results_df = pd.DataFrame(results)
if skipped_train_sizes:
    print('skipped train sizes with no reached repeats:', skipped_train_sizes)
print('summarized train sizes:', reached_train_sizes)
print(f'final peak RSS: {MEMORY_PEAK_MB:.2f} MB')
results_df


   train | repeat |  roc_auc | avg_prec |  seconds |   rss_mb |  peak_mb
-------------------------------------------------------------------------------
     321 |      1 |   0.3998 |   0.4360 |     9.36 |  1251.51 |  1251.51
     642 |      1 |   0.4369 |   0.4584 |    13.25 |  1252.38 |  1252.38
    1285 |      1 |   0.4922 |   0.5345 |    25.51 |  1274.61 |  1274.61
    2570 |      1 |   0.4898 |   0.5093 |    45.32 |  1277.21 |  1277.21


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
frac = 0.8  # Local quadratic LOESS span; adjust between 0 and 1 for more or less smoothing.
x = results_df['train_size'].to_numpy()

plot_series_with_band_loess(
    axes[0],
    x,
    results_df['mean_roc_auc'],
    y_std=results_df['std_roc_auc'],
    frac=frac,
    label='Mean ROC-AUC',
)
axes[0].set_xlabel('Training graphs')
axes[0].set_ylabel('ROC-AUC')
axes[0].set_title('Learning curve: ROC-AUC')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

plot_series_with_band_loess(
    axes[1],
    x,
    results_df['mean_avg_precision'],
    y_std=results_df['std_avg_precision'],
    frac=frac,
    color='tab:green',
    label='Mean average precision',
)
axes[1].set_xlabel('Training graphs')
axes[1].set_ylabel('Average precision')
axes[1].set_title('Learning curve: average precision')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plot_series_with_band_loess(
    axes[2],
    x,
    results_df['mean_runtime_sec'],
    y_std=results_df['std_runtime_sec'],
    frac=frac,
    color='tab:red',
    label='Mean runtime',
)
axes[2].set_xlabel('Training graphs')
axes[2].set_ylabel('Seconds')
axes[2].set_title('End-to-end runtime')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()


## Minimal Full-Stream `partial_fit` Example

For the default HIV setup, `NSPPK.fit(...)` is a no-op because no attribute embedding or clustering must be learned. That means `stream_from(...)` can directly yield supervised `(X_batch, y_batch)` pairs when a `label_extractor` is provided.


In [ ]:
%%time

stream_vectorizer = NSPPK(
    radius=RADIUS,
    distance=DISTANCE,
    connector=CONNECTOR,
    nbits=NBIT,
    dense=False,
    parallel=PARALLEL,
)

stream_model = SGDClassifier(loss='log_loss', alpha=1e-5, penalty='l2', random_state=RANDOM_STATE)
classes = np.array([0, 1])

for batch_idx, (X_batch, y_batch) in enumerate(
    stream_vectorizer.stream_from(
        DATASET_FILE,
        'smiles',
        batch_size=BATCH_SIZE,
        label_extractor=lambda graph: int(graph.graph['HIV_active']),
    )
):
    if batch_idx == 0:
        stream_model.partial_fit(X_batch, y_batch, classes=classes)
    else:
        stream_model.partial_fit(X_batch, y_batch)

stream_model
